# Making list of filenames

In [1]:
import matplotlib.pyplot as plt
from obspy import read_events
from obspy import UTCDateTime
import numpy as np
import pandas as pd
import scipy.stats as stats
import wget
import csv

In [2]:
# Catalog Data
big10_catalog = read_events("Big10_Greece_Seismicity.xml")
filenames = []

j=1
for event in big10_catalog: # For each earthquake

    # Read earthquake data
    origin = event.preferred_origin() or event.origins[0]
    event_time = origin.time
    focal_mech = event.preferred_focal_mechanism() or event.focal_mechanisms[0]
    moment_tensor = focal_mech.moment_tensor
    nodal_planes = focal_mech.nodal_planes
    plane1 = nodal_planes.nodal_plane_1
    plane2 = nodal_planes.nodal_plane_2
    
    mag = event.preferred_magnitude() or event.magnitudes[0]
    mag = float(mag.mag)

    for plane in (plane1,plane2):
        filenames.append(f"Greece_EQ{j}_M{mag}_{plane}_{event_time}.csv")
    j=j+1

    df = pd.DataFrame(filenames)
    df.to_csv("filenames.csv", index=False, header=None)

# Downloading the files

In [3]:
# Catalog Data
filenames = pd.read_csv("filenames.csv", names=['col'], header=None)
filenames=filenames['col'].tolist()
print(filenames)

['Greece_EQ1_M6.72_NodalPlane(strike=201.0, dip=44.0, rake=55.0)_2006-01-08T11:35:00.300000Z.csv', 'Greece_EQ1_M6.72_NodalPlane(strike=66.0, dip=55.0, rake=119.0)_2006-01-08T11:35:00.300000Z.csv', 'Greece_EQ2_M6.85_NodalPlane(strike=332.0, dip=6.0, rake=120.0)_2008-02-14T10:09:29.000000Z.csv', 'Greece_EQ2_M6.85_NodalPlane(strike=121.0, dip=85.0, rake=87.0)_2008-02-14T10:09:29.000000Z.csv', 'Greece_EQ3_M6.54_NodalPlane(strike=337.0, dip=5.0, rake=127.0)_2008-02-14T12:09:02.700000Z.csv', 'Greece_EQ3_M6.54_NodalPlane(strike=120.0, dip=86.0, rake=87.0)_2008-02-14T12:09:02.700000Z.csv', 'Greece_EQ4_M6.76_NodalPlane(strike=339.0, dip=3.0, rake=130.0)_2013-10-12T13:11:56.400000Z.csv', 'Greece_EQ4_M6.76_NodalPlane(strike=119.0, dip=88.0, rake=88.0)_2013-10-12T13:11:56.400000Z.csv', 'Greece_EQ5_M6.86_NodalPlane(strike=73.0, dip=85.0, rake=-177.0)_2014-05-24T09:25:18.800000Z.csv', 'Greece_EQ5_M6.86_NodalPlane(strike=343.0, dip=87.0, rake=-5.0)_2014-05-24T09:25:18.800000Z.csv', 'Greece_EQ6_M6.5_N

In [4]:
EQ=1
for i in range(20):
    filename=filenames[i]
    print(EQ)
    print(f"GNSSVerify/EQ{EQ}.{2-(i+1)%2}/")
    print(filename)
    relevant_stations = pd.read_csv(f"Finite/{filename}")
    sta_ids = relevant_stations["Station_ID"]
    for station in sta_ids:
        url = f"https://geodesy.unr.edu/gps_timeseries/IGS20/tenv3/EU/{station.upper()}.EU.tenv3"
        wget.download(url, out=f"GNSSVerify/EQ{EQ}.{2-(i+1)%2}/")
    if ((i+1)%2==0):
        EQ=EQ+1

1
GNSSVerify/EQ1.1/
Greece_EQ1_M6.72_NodalPlane(strike=201.0, dip=44.0, rake=55.0)_2006-01-08T11:35:00.300000Z.csv
100% [....................................................] 300696 / 3006961
GNSSVerify/EQ1.2/
Greece_EQ1_M6.72_NodalPlane(strike=66.0, dip=55.0, rake=119.0)_2006-01-08T11:35:00.300000Z.csv
100% [....................................................] 300696 / 3006962
GNSSVerify/EQ2.1/
Greece_EQ2_M6.85_NodalPlane(strike=332.0, dip=6.0, rake=120.0)_2008-02-14T10:09:29.000000Z.csv
100% [....................................................] 608736 / 6087362
GNSSVerify/EQ2.2/
Greece_EQ2_M6.85_NodalPlane(strike=121.0, dip=85.0, rake=87.0)_2008-02-14T10:09:29.000000Z.csv
100% [....................................................] 608736 / 6087363
GNSSVerify/EQ3.1/
Greece_EQ3_M6.54_NodalPlane(strike=337.0, dip=5.0, rake=127.0)_2008-02-14T12:09:02.700000Z.csv
100% [....................................................] 608736 / 6087363
GNSSVerify/EQ3.2/
Greece_EQ3_M6.54_NodalPlane(st

# Computing Coseismic Deformation from GNSS time-series

In [11]:
%%bash

t1_list=( # A year before event
    None
    2005.0192 # 2006-01-08
    2007.1202 # 2008-02-14
    2007.1202 # 2008-02-14
    2012.7781 # 2013-10-12
    2013.3918 # 2014-05-24
    2014.8767 # 2015-11-17
    2016.5479 # 2017-07-20
    2017.8137 # 2018-10-25
    2019.3333 # 2020-05-02
    2019.8279 # 2020-10-30
)

t2_list=( # A year after event
    None
    2007.0192 # 2006-01-08
    2009.1202 # 2008-02-14
    2009.1202 # 2008-02-14
    2014.7781 # 2013-10-12
    2015.3918 # 2014-05-24
    2016.8767 # 2015-11-17
    2018.5479 # 2017-07-20
    2019.8137 # 2018-10-25
    2021.3333 # 2020-05-02
    2021.8279 # 2020-10-30
)

dirEXE=/mnt/c/Users/Bea/Downloads/Okada/Refining_Thesis/GNSSVerify # Contains Executables
cd $dirEXE # Go to executables folder

for i in $(seq 1 10); do
    t1=${t1_list[$i]}
    t2=${t2_list[$i]}

    for j in $(seq 1 2); do
        dirIN="${dirEXE}/EQ${i}.${j}" # Input Folder
        dirOUT="${dirIN}" # Output Folder

        files=("$dirIN"/*.EU.tenv3)

        if [ ! -e "${files[0]}" ]; then
            continue
        fi

        # For every .tenv3 file in the folder
        for f in "${files[@]}"; do
            sta=$(basename "$f" .EU.tenv3) # Station Code/ Identifier
            rm -f serv.inp serv.p_tau serv.bayes # Delete residual files from prev run
        
            # Reads file "$f", filters in range [t1,t2], and extracts 3:Decimal Year and 9:East to serv.inp
            gawk -v tin="$t1" -v tout="$t2" -v c=9 'NR>1 && $3>=tin && $3<=tout {print $3, $c}' "$f" > serv.inp
            if [ -s serv.inp ]; then
                octave -q < input_cycleslip.m
                [ -f serv.bayes ] && cp serv.bayes "$dirOUT/$sta.E.bayes.out"
                [ -f serv.p_tau ] && cp serv.p_tau "$dirOUT/$sta.E.bayes.p_tau"
            else
                echo "Warning: No data for $sta in window $t1 - $t2"
            fi
            
            rm -f serv.inp serv.p_tau serv.bayes # Delete residual files from prev run
        
            # Reads file "$f", filters in range [t1,t2], and extracts 3:Decimal Year and 11:North to serv.inp
            gawk -v tin="$t1" -v tout="$t2" -v c=11 'NR>1 && $3>=tin && $3<=tout {print $3, $c}' "$f" > serv.inp
            if [ -s serv.inp ]; then
                octave -q < input_cycleslip.m
                [ -f serv.bayes ] && cp serv.bayes "$dirOUT/$sta.N.bayes.out"
                [ -f serv.p_tau ] && cp serv.p_tau "$dirOUT/$sta.N.bayes.p_tau"
            else
                echo "Warning: No data for $sta in window $t1 - $t2"
            fi
            
            rm -f serv.inp serv.p_tau serv.bayes # Delete residual files from prev run
        
            # Reads file "$f", filters in range [t1,t2], and extracts 3:Decimal Year and 13:Up to serv.inp
            gawk -v tin="$t1" -v tout="$t2" -v c=13 'NR>1 && $3>=tin && $3<=tout {print $3, $c}' "$f" > serv.inp
            if [ -s serv.inp ]; then
                octave -q < input_cycleslip.m
                [ -f serv.bayes ] && cp serv.bayes "$dirOUT/$sta.U.bayes.out"
                [ -f serv.p_tau ] && cp serv.p_tau "$dirOUT/$sta.U.bayes.p_tau"
            else
                echo "Warning: No data for $sta in window $t1 - $t2"
            fi

            rm -f serv.inp serv.p_tau serv.bayes
        done
    done
done

tau0 = 2006.182100000000
tau0 = 2006.225900000000
tau0 = 2005.385400000000
tau0 = 2006.157400000000
tau0 = 2005.850800000000
tau0 = 2005.804200000000
tau0 = 2006.069800000000
tau0 = 2006.064300000000
tau0 = 2006.310700000000
tau0 = 2006.685800000000
tau0 = 2006.759800000000
tau0 = 2006.721400000000
tau0 = 2006.080800000000
tau0 = 2006.105400000000
tau0 = 2005.796000000000
tau0 = 2006.160200000000
tau0 = 2005.158100000000
tau0 = 2006.338100000000
tau0 = 2006.458600000000
tau0 = 2006.458600000000
tau0 = 2006.850100000000
tau0 = 2006.094500000000
tau0 = 2005.845300000000
tau0 = 2006.381900000000
tau0 = 2006.091700000000
tau0 = 2005.122500000000
tau0 = 2005.368900000000
tau0 = 2006.086200000000
tau0 = 2006.099900000000
tau0 = 2006.368200000000
tau0 = 2006.116400000000
tau0 = 2005.232000000000
tau0 = 2006.316200000000
tau0 = 2006.718700000000
tau0 = 2006.442200000000
tau0 = 2006.762500000000
tau0 = 2006.707700000000
tau0 = 2006.833700000000
tau0 = 2006.743300000000
tau0 = 2006.696800000000


error: t(2): out of bound 1 (dimensions are 1x1)
error: called from
    posterior_cont_gen at line 36 column 4
error: t(2): out of bound 1 (dimensions are 1x1)
error: called from
    posterior_cont_gen at line 36 column 4
error: t(2): out of bound 1 (dimensions are 1x1)
error: called from
    posterior_cont_gen at line 36 column 4


tau0 = 2013.642700000000
tau0 = 2013.642700000000
tau0 = 2013.850800000000
tau0 = 2013.100600000000
tau0 = 2014.414800000000
tau0 = 2013.798800000000
tau0 = 2013.711200000000
tau0 = 2013.601600000000
tau0 = 2014.097200000000


error: t(2): out of bound 1 (dimensions are 1x1)
error: called from
    posterior_cont_gen at line 36 column 4
error: t(2): out of bound 1 (dimensions are 1x1)
error: called from
    posterior_cont_gen at line 36 column 4
error: t(2): out of bound 1 (dimensions are 1x1)
error: called from
    posterior_cont_gen at line 36 column 4


tau0 = 2014.581800000000
tau0 = 2014.694000000000
tau0 = 2014.381900000000
tau0 = 2015.137600000000
tau0 = 2015.071900000000
tau0 = 2015.165000000000
tau0 = 2014.590000000000
tau0 = 2015.014400000000
tau0 = 2014.335400000000
tau0 = 2015.170400000000
tau0 = 2013.645400000000
tau0 = 2014.351800000000
tau0 = 2015.151300000000
tau0 = 2015.203300000000
tau0 = 2014.371000000000
tau0 = 2013.511300000000
tau0 = 2014.313500000000
tau0 = 2014.346300000000
tau0 = 2014.165600000000
tau0 = 2013.749500000000
tau0 = 2014.811800000000
tau0 = 2015.140300000000
tau0 = 2013.752200000000
tau0 = 2014.351800000000
tau0 = 2013.749500000000
tau0 = 2013.749500000000
tau0 = 2014.371000000000
tau0 = 2013.749500000000
tau0 = 2013.574300000000
tau0 = 2013.957600000000
tau0 = 2015.170400000000
tau0 = 2014.833700000000
tau0 = 2014.880200000000
tau0 = 2014.395600000000
tau0 = 2014.395600000000
tau0 = 2014.373700000000
tau0 = 2013.634500000000
tau0 = 2013.574300000000
tau0 = 2014.324400000000
tau0 = 2014.168400000000


...

tau0 = 2021.522200000000


    posterior_cont_gen at line 183 column 15

error: 'muk' undefined near line 205, column 7
error: called from
    posterior_cont_gen at line 205 column 6


tau0 = 2021.522200000000


    posterior_cont_gen at line 183 column 15

error: 'muk' undefined near line 205, column 7
error: called from
    posterior_cont_gen at line 205 column 6


tau0 = 2020.219000000000
tau0 = 2020.339500000000
tau0 = 2020.810400000000
tau0 = 2021.793300000000
tau0 = 2021.787800000000
tau0 = 2021.790600000000
tau0 = 2020.167000000000
tau0 = 2020.829600000000
tau0 = 2020.802200000000
tau0 = 2020.117700000000
tau0 = 2020.846000000000
tau0 = 2020.799500000000
tau0 = 2020.208100000000
tau0 = 2021.388100000000
tau0 = 2020.810400000000
tau0 = 2020.216300000000
tau0 = 2021.388100000000
tau0 = 2020.802200000000
tau0 = 2020.769300000000
tau0 = 2020.832300000000
tau0 = 2020.900800000000
tau0 = 2020.840500000000
tau0 = 2020.840500000000
tau0 = 2020.840500000000
tau0 = 2020.763900000000
tau0 = 2020.637900000000
tau0 = 2020.818600000000
tau0 = 2020.772100000000
tau0 = 2020.826800000000
tau0 = 2020.944600000000
tau0 = 2020.219000000000
tau0 = 2020.632400000000
tau0 = 2020.739200000000
tau0 = 2020.514700000000
tau0 = 2020.509200000000
tau0 = 2020.509200000000
tau0 = 2020.599600000000
tau0 = 2020.646100000000
tau0 = 2020.763900000000
tau0 = 2020.783000000000


## Read the displacement corresponding to each earthquake and write to file

In [12]:
%%bash

events=(
    None
    2006.0192 # 2006-01-08
    2008.1202 # 2008-02-14
    2008.1202 # 2008-02-14
    2013.7781 # 2013-10-12
    2014.3918 # 2014-05-24
    2015.8767 # 2015-11-17
    2017.5479 # 2017-07-20
    2018.8137 # 2018-10-25
    2020.3333 # 2020-05-02
    2020.8279 # 2020-10-30
)

dirEXE=/mnt/c/Users/Bea/Downloads/Okada/Refining_Thesis/GNSSVerify # Contains Executables
disp=(
    "E"
    "N"
    "U"
)

cd ${dirEXE} || exit 1

for i in $(seq 1 10); do # For each quake
    event=${events[$i]}

    for j in $(seq 1 2); do # For each nodal plane
        dirIN=${dirEXE}/EQ${i}.${j} # Input Folder
        dirOUT=${dirIN} # Output Folder
        echo "EQ${i}.${j} : $event" | tee -a GNSSVerify.txt

        for g in ${disp[@]}; do # For each component of displacement
            files=("$dirIN"/*."${g}".bayes.p_tau)

            if [ ! -e "${files[0]}" ]; then
                continue
            fi

            echo ${g} | tee -a GNSSVerify.txt
    
            # For every .p_tau file
            for f in "${files[@]}"; do
                sta=$(basename "$f" ."${g}".bayes.p_tau) # Station Code/ Identifier
                gawk -v sta="$sta" -v val="${event}" -v OFS="\t" '
                    $1 >= val {
                        print sta, $1, $3
                        found = 1
                        exit
                    }
                    END {
                        if (!found) print sta, "NO_DATA_AFTER_EVENT", "N/A"
                    }
                ' "$f" | tee -a GNSSVerify.txt
            done 
        done
    done
done

EQ1.1 : 2006.0192
E
AKYR	2006.17110	-0.00632
ANOP	2006.02050	0.00252
ATRS	2006.02050	0.00854
GVDS	2006.48600	0.00000
KERY	2006.02050	0.00604
KITH	2006.02050	0.00229
KOUN	2006.45860	-0.01597
KRYO	2006.02050	0.00604
MEN1	2006.02050	0.00357
MET4	2006.02050	0.00492
NEA1	2006.02050	0.00349
PSAR	2006.44220	-0.02388
RLSO	2006.57630	0.00000
SPR2	2006.55170	0.00000
TUC2	2006.02050	0.00337
VASS	2006.02050	0.00465
XRSO	2006.02050	0.00297
N
AKYR	2006.17110	-0.01249
ANOP	2006.02050	0.00032
ATRS	2006.02050	-0.00339
GVDS	2006.48600	0.00000
KERY	2006.02050	-0.00119
KITH	2006.02050	-0.00002
KOUN	2006.45860	-0.02299
KRYO	2006.02050	0.00052
MEN1	2006.02050	-0.00000
MET4	2006.02050	-0.00103
NEA1	2006.02050	0.00080
PSAR	2006.44220	-0.01712
RLSO	2006.57630	0.00000
SPR2	2006.55170	0.00000
TUC2	2006.02050	0.00239
VASS	2006.02050	-0.00223
XRSO	2006.02050	-0.00063
U
AKYR	2006.17110	-0.00228
ANOP	2006.02050	-0.00407
ATRS	2006.02050	0.00005
GVDS	2006.48600	0.00000
KERY	2006.02050	-0.00013
KITH	2006.02050	0.00303


## Read the date of the detected cycle slip and write to file

In [13]:
%%bash

dirEXE=/mnt/c/Users/Bea/Downloads/Okada/Refining_Thesis/GNSSVerify # Contains Executables
disp=(
    "E"
    "N"
    "U"
)

cd ${dirEXE} || exit 1

for i in $(seq 1 10); do # For each quake

    for j in $(seq 1 2); do # For each nodal plane
        dirIN=${dirEXE}/EQ${i}.${j} # Input Folder
        dirOUT=${dirIN} # Output Folder
        echo EQ${i}.${j} | tee -a CycleSlip.txt

        for g in ${disp[@]}; do # For each component of displacement
            files=("$dirIN"/*."${g}".bayes.out)
            if [ ! -e "${files[0]}" ]; then
                continue
            fi
            echo ${g} | tee -a CycleSlip.txt
    
            # For every .bayes.out file
            for f in "${files[@]}"; do
                sta=$(basename "$f" ."${g}".bayes.out) # Station Code/ Identifier
                gawk -v sta="$sta" -v OFS="\t" '
                    /tau0/ && /=/ {
                        print sta, $NF
                        found = 1
                        exit
                    }
                    END {
                    }
                ' "$f" | tee -a CycleSlip.txt
            done 
        done
    done
done

EQ1.1
E
AKYR	2006.1821
ANOP	2006.1574
ATRS	2006.0698
GVDS	2006.6858
KERY	2006.0808
KITH	2006.1602
KOUN	2006.4586
KRYO	2006.0945
MEN1	2006.0917
MET4	2006.0862
NEA1	2006.1164
PSAR	2006.7187
RLSO	2006.7077
SPR2	2006.6968
TUC2	2006.0917
VASS	2005.9877
XRSO	2006.7023
N
AKYR	2006.2259
ANOP	2005.8508
ATRS	2006.0643
GVDS	2006.7598
KERY	2006.1054
KITH	2005.1581
KOUN	2006.4586
KRYO	2005.8453
MEN1	2005.1225
MET4	2006.0999
NEA1	2005.2320
PSAR	2006.4422
RLSO	2006.8337
SPR2	2006.7515
TUC2	2006.0096
VASS	2006.1027
XRSO	2006.1629
U
AKYR	2005.3854
ANOP	2005.8042
ATRS	2006.3107
GVDS	2006.7214
KERY	2005.7960
KITH	2006.3381
KOUN	2006.8501
KRYO	2006.3819
MEN1	2005.3689
MET4	2006.3682
NEA1	2006.3162
PSAR	2006.7625
RLSO	2006.7433
SPR2	2006.7214
TUC2	2005.7960
VASS	2005.7906
XRSO	2006.3135
EQ1.2
E
AKYR	2006.1821
ANOP	2006.1574
ATRS	2006.0698
GVDS	2006.6858
KERY	2006.0808
KITH	2006.1602
KOUN	2006.4586
KRYO	2006.0945
MEN1	2006.0917
MET4	2006.0862
NEA1	2006.1164
PSAR	2006.7187
RLSO	2006.7077
SPR2	2006.6968
TRIZ	

## Saving Cycle Slip and Respective Coseismic Displacement and A Posterior Probability to One File

In [7]:
%%bash

dirEXE=/mnt/c/Users/Bea/Downloads/Okada/Refining_Thesis/GNSSVerify # Contains Executables
cd ${dirEXE} || exit 1
disp=("E" "N" "U")

for i in $(seq 1 10); do # For each quake
    for j in $(seq 1 2); do # For each nodal plane
        dirIN="${dirEXE}/EQ${i}.${j}" # Input Folder
        dirOUT="${dirIN}" # Output Folder
        echo EQ${i}.${j} | tee -a GNSSData.txt # Save which quake and nodal plane

        for g in "${disp[@]}"; do # For each component of displacement
            files=("$dirIN"/*."${g}".bayes.out) # Contains Cycle Slip Date Detection
            if [ ! -e "${files[0]}" ]; then
                continue
            fi
            echo ${g} | tee -a GNSSData.txt # Save which component
    
            # For every file
            for f in "${files[@]}"; do
                sta=$(basename "$f" ."${g}".bayes.out) # Station Code/ Identifier
                h="$dirIN/$sta.${g}.bayes.p_tau" # Contains Coseismic Slip in mm
                cycle=$(gawk '/tau0/ && /=/ { print $NF; exit }' "$f") # Save Cycle Slip Date in cycle
                coseismic="N/A"
                if [ -n "$cycle" ] && [ -f "$h" ]; then
                    coseismic=$(gawk -v OFS="\t" -v cycle="$cycle" '
                        sprintf("%.5f", $1) == sprintf("%.5f", cycle) {
                            print $3, $2
                            exit
                        }
                    ' "$h")
                fi
                echo -e "${sta}\t${cycle:-N/A}\t${coseismic:-N/A}" | tee -a GNSSData.txt
            done 
        done
    done
done

EQ1.1
E
AKYR	2006.1821	-0.00641	0.82546
ANOP	2006.1574	0.00361	0.45542
ATRS	2006.0698	0.00939	0.59455
GVDS	2006.6858	-0.00175	0.28261
KERY	2006.0808	0.00622	0.22616
KITH	2006.1602	0.00398	0.55318
KOUN	2006.4586	-0.01597	0.84926
KRYO	2006.0945	0.00690	0.47895
MEN1	2006.0917	0.00419	0.30135
MET4	2006.0862	0.00536	0.42696
NEA1	2006.1164	0.00456	0.65578
PSAR	2006.7187	-0.02384	0.82556
RLSO	2006.7077	-0.00160	0.35591
SPR2	2006.6968	-0.00239	0.05175
TUC2	2006.0917	0.00449	0.27959
VASS	2005.9877	0.00483	0.35897
XRSO	2006.7023	-0.00308	0.29420
N
AKYR	2006.2259	-0.01267	0.99965
ANOP	2005.8508	0.00220	0.18475
ATRS	2006.0643	-0.00382	0.80052
GVDS	2006.7598	0.00120	0.07670
KERY	2006.1054	-0.00270	0.27521
KITH	2005.1581	-0.00170	0.36487
KOUN	2006.4586	-0.02299	0.99999
KRYO	2005.8453	0.00115	0.23261
MEN1	2005.1225	-0.00253	0.13472
MET4	2006.0999	-0.00194	0.20831
NEA1	2005.2320	-0.00145	0.08042
PSAR	2006.4422	-0.01712	0.99998
RLSO	2006.8337	-0.00166	0.48090
SPR2	2006.7515	0.00150	0.08290
TUC2	2006.00

## Saving only the Cycle Slip of the files with probability 100%

In [8]:
%%bash

dirEXE=/mnt/c/Users/Bea/Downloads/Okada/Refining_Thesis/GNSSVerify # Contains Executables
cd ${dirEXE} || exit 1
disp=("E" "N" "U")

for i in $(seq 1 10); do # For each quake
    for j in $(seq 1 2); do # For each nodal plane
        dirIN="${dirEXE}/EQ${i}.${j}" # Input Folder
        dirOUT="${dirIN}" # Output Folder
        echo EQ${i}.${j} | tee -a GNSS100.txt # Save which quake and nodal plane

        for g in "${disp[@]}"; do # For each component of displacement
            files=("$dirIN"/*."${g}".bayes.out) # Contains Cycle Slip Date Detection
            if [ ! -e "${files[0]}" ]; then
                continue
            fi
            echo ${g} | tee -a GNSS100.txt # Save which component
    
            # For every file
            for f in "${files[@]}"; do
                sta=$(basename "$f" ."${g}".bayes.out) # Station Code/ Identifier
                h="$dirIN/$sta.${g}.bayes.p_tau" # Contains Coseismic Slip in mm
                cycle=$(gawk '/tau0/ && /=/ { print $NF; exit }' "$f") # Save Cycle Slip Date in cycle
                coseismic="N/A"
            
                if [ -n "$cycle" ] && [ -f "$h" ]; then
                    # AWK only extracts and prints $3 (slip) and $2 (prob) to stdout
                    coseismic=$(gawk -v OFS="\t" -v cycle="$cycle" '
                        (sprintf("%.5f", $1) == sprintf("%.5f", cycle)) && (sprintf("%.5f", $2) == "1.00000") {
                            print $3, $2
                            exit
                        }
                    ' "$h")
                fi
            
                # Bash handles printing and appending to the file ONLY if a match was found
                if [ -n "$coseismic" ]; then
                    echo -e "${sta}\t${cycle:-N/A}\t${coseismic}" | tee -a GNSS100.txt
                fi
            done
        done
    done
done

EQ1.1
E
N
U
EQ1.2
E
N
U
EQ2.1
E
N
U
EQ2.2
E
N
U
EQ3.1
E
N
U
EQ3.2
E
N
U
EQ4.1
E
N
U
EQ4.2
E
N
U
EQ5.1
E
KAL2	2014.8255	0.00828	1.00000
LEMN	2014.3956	-0.02286	1.00000
N
LEMN	2014.3956	-0.05069	1.00000
U
ROZH	2014.8392	-0.04379	1.00000
EQ5.2
E
KAL2	2014.8255	0.00828	1.00000
LEMN	2014.3956	-0.02286	1.00000
N
LEMN	2014.3956	-0.05069	1.00000
SVRT	2014.4832	0.01682	1.00000
U
ROZH	2014.8392	-0.04379	1.00000
EQ6.1
E
LFKD	2015.8850	-0.03668	1.00000
PONT	2015.8768	-0.20771	1.00000
SPAN	2015.8768	-0.07704	1.00000
N
LFKD	2015.8850	-0.03372	1.00000
PONT	2015.8768	-0.38469	1.00000
SISS	2016.1068	-0.02551	1.00000
SPAN	2015.8768	-0.06462	1.00000
U
PONT	2015.8768	-0.06413	1.00000
TRIZ	2016.1725	-0.29878	1.00000
EQ6.2
E
LFKD	2015.8850	-0.03668	1.00000
PONT	2015.8768	-0.20771	1.00000
SPAN	2015.8768	-0.07704	1.00000
N
LFKD	2015.8850	-0.03372	1.00000
PONT	2015.8768	-0.38469	1.00000
SISS	2016.1068	-0.02551	1.00000
SPAN	2015.8768	-0.06462	1.00000
U
PONT	2015.8768	-0.06413	1.00000
TRIZ	2016.1725	-0.29878	1.0